# neural density estimation --- MADE
We will implement the **masked autoencoder for density estimation (MADE)**. 

As we discussed, MADE is a simple autoencoder combined with masks that enforce the autoregressive properties. We'll generally follow the approach from *Germain et al.*(2015) and implement using `pytorch`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np 
import matplotlib.pyplot as plt
import corner

In [ ]:
torch.manual_seed(0)

In [ ]:
if torch.cuda.is_available(): 
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")
print("Using device:", device)

# target distribution $p(x)$
Let's start by making some complicated and interesting target distribution $p(x)$. 

In [ ]:
def make_synthetic_data(N, device=device):
    ''' complicated 4D distribution: 
    - (x0, x1): 3-component Gaussian mixture with different means & covariances
    - x2: nonlinear function of (x0, x1) + noise
          x2 = sin(x0) + 0.5 * x1^2 + eps
    - x3: heavy-tailed (Student-t-like) variable
    '''
    mus = torch.stack([
        torch.tensor([0.0, 0.0], device=device),
        torch.tensor([3.0, -3.0], device=device),
        torch.tensor([-3.0, 3.0], device=device)]) 

    covs = []
    covs.append(torch.tensor([[1.0, 0.2],
                              [0.2, 1.0]], device=device))
    covs.append(torch.tensor([[2.0, 1.5],
                              [1.5, 2.0]], device=device))
    covs.append(torch.tensor([[0.5, -0.3],
                              [-0.3, 0.5]], device=device))
    covs = torch.stack(covs, dim=0)  
    pis = torch.tensor([0.4, 0.35, 0.25], device=device)

    comp_idx = torch.multinomial(pis, num_samples=N, replacement=True)  # [N]

    # For each sample, get its mean and covariance
    mu = mus[comp_idx]           # [N, 2]
    cov = covs[comp_idx]         # [N, 2, 2]

    # Sample from N(0, I) and transform via Cholesky of cov
    eps = torch.randn(N, 2, device=device)  # [N, 2]
    # Cholesky factor per component
    L = torch.linalg.cholesky(cov)          # [N, 2, 2]
    xy = (L @ eps.unsqueeze(-1)).squeeze(-1) + mu  # [N, 2]

    x0 = xy[:, 0]
    x1 = xy[:, 1]

    # ----- Nonlinear, noisy x2 -----
    # x2 = sin(x0) + 0.5 * x1^2 + noise
    noise_x2 = 0.5 * torch.randn(N, device=device)
    x2 = torch.sin(x0) + 0.5 * x1**2 + noise_x2

    # ----- Heavy-tailed x3 (Student-t like) -----
    # t-dist ~ z / sqrt(u/ν), with z ~ N(0,1), u ~ χ²(ν)
    nu = 3.0
    z = torch.randn(N, device=device)
    u = torch.distributions.Chi2(df=torch.tensor(nu, device=device)).sample((N,))
    x3 = z * torch.sqrt(nu / u)  # Student-t(ν)

    return torch.stack([x0, x1, x2, x3], dim=-1)

In [ ]:
samples = make_synthetic_data(10000, device='cpu')
fig = plt.figure(figsize=(6,6))
fig = corner.corner(samples.numpy(), range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)

Looks difficult enough that a Gaussian mixture model or other density estimation methods would struggle.

# implementating MADE
Lets start by creating the masks that enforce the autoregressive properties on the encoder

In [ ]:
def create_masks(hidden_dims, ndim=4, device="cpu"):
    # indices for input layer
    input_indices = torch.arange(1, ndim + 1, device=device)

    # indices for hidden layers in [1, ndim - 1]
    indices = []
    for h in hidden_dims:
        # for each hidden unit, sample an integer index in [1, ndim-1]
        ind = torch.randint(1, ndim, size=(h,), device=device)
        indices.append(ind)

    output_indices = torch.arange(1, ndim + 1, device=device).repeat_interleave(2) # [1, 1, 2, 2, ...]

    # ndim-dimensional input --> mu, \log sigma per dimension
    layer_sizes = [ndim] + hidden_dims + [2 * ndim]
    all_indices = [input_indices] + indices + [output_indices]

    masks = []
    for l in range(len(layer_sizes) - 1):
        i_in = all_indices[l]      
        i_out = all_indices[l + 1] 
        if l < len(layer_sizes) - 2:
            # hidden layers: allow if i_out >= i_in
            mask = (i_out[:, None] >= i_in[None, :]).float()
        else:
            # output layer: allow if i_out > i_in
            mask = (i_out[:, None] > i_in[None, :]).float()
        masks.append(mask)
    return masks

Now we can incorporate the masks into a MADE model

In [ ]:
class MADE(nn.Module):
    def __init__(self, hidden_dims, D=4, device="cpu"):
        super().__init__()
        self.D = D
        self.hidden_dims = hidden_dims
        self.device = device
        
        layer_sizes = [D] + hidden_dims + [D * 2]

        layers = []
        for in_f, out_f in zip(layer_sizes[:-1], layer_sizes[1:]):
            layers.append(nn.Linear(in_f, out_f))
        self.layers = nn.ModuleList(layers)

        # build masks
        masks = create_masks(hidden_dims, ndim=D, device=self.device)
        # so they move with .to(device), but are not trainable
        self.masks = nn.ParameterList(
            [nn.Parameter(m, requires_grad=False) for m in masks]
        )
        
        # initialize W and b
        for layer in self.layers:
            nn.init.normal_(layer.weight, mean=0.0, std=1.0 / np.sqrt(layer.in_features))
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        '''forward pass 
        '''
        h = x
        L = len(self.layers)
        for l, layer in enumerate(self.layers):
            # apply autoregressive mask to weights
            masked_weight = layer.weight * self.masks[l]
            h = F.linear(h, masked_weight, layer.bias)
            if l < L - 1:
                h = F.relu(h)
                
        B = h.shape[0]
        h = h.view(B, self.D, 2)
        mean = h[..., 0]
        log_std = h[..., 1]
        return mean, log_std

    def log_prob(self, x):
        ''' log p(x)
        '''
        mean, log_std = self.forward(x)
        inv_std = torch.exp(-log_std)
        z = (x - mean) * inv_std
        log_det = -log_std
        logp_per_dim = -0.5 * (np.log(2.0 * np.pi) + z**2) + log_det
        return torch.sum(logp_per_dim, dim=-1)
    
    def sample(self, num_samples):
        self.eval()

        # sequentially sample x
        x = torch.zeros(num_samples, self.D, device=self.device)
        for i in range(self.D):
            mean, log_std = self.forward(x)
            std = torch.exp(log_std)
            eps = torch.randn(num_samples, device=self.device)
            x[:, i] = mean[:, i] + std[:, i] * eps
        return x

Now lets train a MADE to estimate this gnarly distribution

In [ ]:
hidden_dims = [128, 64] # architecture of the encoder
lr = 1e-3 # learning rate 
batch_size = 128 # batch size
num_epochs = 100 # number of training epochs

In [ ]:
# MADE model
model = MADE(hidden_dims, device=device).to(device)

In [ ]:
# initialize the ADAM optimzer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
x_train = samples.to(device) 
N_train = samples.shape[0]

num_batches = N_train // batch_size
train_losses = []
for epoch in range(1, num_epochs + 1):
    # shuffle train data at each epoch
    perm = torch.randperm(N_train, device=device)
    x_train = x_train[perm]

    model.train()
    epoch_loss = 0.0
    
    for i in range(num_batches):
        batch_x = x_train[i * batch_size:(i + 1) * batch_size]
        
        optimizer.zero_grad()
        lp = model.log_prob(batch_x)     
        loss = -lp.mean()                  
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= num_batches
    train_losses.append(epoch_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: train log p = {epoch_loss:.4f}")

We won't go through a validation step to check for overfitting and etc here but you should always validate the training!

Lets check how well $q_\phi(\theta, \mathcal{D})$ approximates $p(\theta, \mathcal{D})$. 

In [ ]:
# draw samples from q_phi
model.eval()
with torch.no_grad():
    q_samples = model.sample(10000).cpu()

In [ ]:
fig = plt.figure(figsize=(6,6))
fig = corner.corner(samples.numpy(), hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)
fig = corner.corner(q_samples.numpy(), color='C1', hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)

Not too bad! 

# In-class exercise: a MADE surrogate for the IMF slope
The exercise in the last lecture ended with a failure. 
With a broad prior on the IMF slope, $\alpha\sim\mathcal{U}(1.5, 3.5)$, the multivariate Gaussian surrogate $q_\phi(\theta, \mathcal{D})$ could not follow the simulations, and the posteriors it gave were too wide for some clusters and biased for others. 

In this exercise we replace the Gaussian with a MADE in the same SBI setup but on a version of the IMF problem that is harder for a Gaussian.

### setup: 

**simulator**: The simulator is the same as previous lectures: masses from $p(m\,|\,\alpha)\propto m^{-\alpha}$ between 1 and 20 $M_\odot$ **but for a cluster of 1000 stars**. Use `simulate_masses`

**data**: As data or observable we use two summaries, different than before: 

$$\mathcal{D} = (s_1, s_2)$$

* $s_1$ = geometric-mean mass $e^{\overline{\log m}}$
* $s_2$ = mass of the most massive star

Use `summaries`

**prior**: For the prior, take a broad Gaussian: $\alpha\sim\mathcal{N}(2.35, 0.4^2)$.

**observations**: We observe four clusters, with $\alpha_{\rm true}$ = 1.7, 2.2, 2.7 and 3.2. Use `D_obs` dict we'll define below. 

The exact posterior is still available in this toy example but in practice SBI allows you to conduct Bayesian inference in situations where the exact posterior is intractable (`exact_posterior`).

In [ ]:
rng = np.random.default_rng(21)
torch.manual_seed(21)

M_MIN, M_MAX, N_STAR = 1.0, 20.0, 1000
PRIOR_MU, PRIOR_SD = 2.35, 0.4 # prior: alpha ~ N(2.35, 0.4^2)

def simulate_masses(alpha, rng, n=N_STAR):
    ''' One simulated cluster of n stellar masses per value of alpha (rows).
    '''
    alpha = np.atleast_1d(alpha)[:, None]
    e = 1 - alpha
    u = rng.uniform(size=(alpha.shape[0], n))
    return (M_MIN**e + u * (M_MAX**e - M_MIN**e))**(1 / e)

def summaries(m):
    ''' D = (geometric-mean mass, most massive star) [M_sun] for each row of masses.
    '''
    return np.column_stack([np.exp(np.log(m).mean(1)), m.max(1)])

a_grid = np.linspace(1.05, 4.0, 3001)

def exact_posterior(m):
    """Exact posterior of alpha on a_grid."""
    e = 1 - a_grid
    lp = len(m) * np.log(e / (M_MAX**e - M_MIN**e)) - a_grid * np.log(m).sum() - 0.5 * ((a_grid - PRIOR_MU) / PRIOR_SD)**2
    p = np.exp(lp - lp.max())
    return p / np.trapezoid(p, a_grid)

def mean_std(p):
    mu = np.trapezoid(a_grid * p, a_grid)
    return mu, np.sqrt(np.trapezoid((a_grid - mu)**2 * p, a_grid))

alpha_obs = [1.7, 2.2, 2.7, 3.2]
D_obs, post_exact = {}, {}
for a_t in alpha_obs:
    m = simulate_masses(a_t, rng)[0]
    D_obs[a_t] = summaries(m[None])[0]
    post_exact[a_t] = exact_posterior(m)
    print(f'alpha_true = {a_t}: exact posterior {mean_std(post_exact[a_t])[0]:.3f} +/- {mean_std(post_exact[a_t])[1]:.3f}')

### 1. sample $p(\alpha, \mathcal{D})$
Sample the joint parameter, data distribution, $p(\alpha, \mathcal{D})$, using the prior and simulator. 

In [ ]:
N_SIM = 5000

### 2. Train a MADE surrogate for $p(\alpha, \mathcal{D})$
Use the `N_SIM` samples from $p(\alpha, \mathcal{D})$ to train a `MADE` surrogate distribution by minimizing the KL divergence (i.e., $\sum[-\log q_\phi(\theta', \mathcal{D}')]$). 
Plot the training and validation loss curves. Use a 80/10/10 split. 


*hint 1: don't forget to standardize/whiten $\alpha$ and $\mathcal{D}$*

*hint 2: write the training as a function so you can rerun it with other settings*

In [ ]:
def train_made(X, hidden_dims=[64, 64], lr=1e-3, batch_size=128, max_epochs=500, patience=20):
    ''' Train a MADE on the rows of X = np.column_stack([alpha_sim, D_sim]). 
    Return the model, the standardization, and the loss history.
    '''
    return 


### 3. Fit a multivariate Gaussian surrogat for $p(\alpha, \mathcal{D})$
As in the last lecture notebook, fit a multivariate Gaussian surrogate for $p(\alpha, \mathcal{D})$. 
Then, compare samples from the MADE and multivariate Gaussian surrogates against the `N_SIM` simulations. Use `corner`. 

### 4. ABC with the MADE surrogate
For each observed cluster, derive the posterior, $p(\alpha|\mathcal{D})$ using ABC with the MADE surrogate. 
Use the distance criteria  $\rho(\mathcal{D}', \mathcal{D}_{\rm obs}) < 0.05$ (distance in units of each summary's standard deviation over the simulations). 
Compare with the posterior you get from doing the same ABC procedure using the Gaussian
surrogate and the exact posterior.

### (Optional) Repeat the MADE posterior using `N_SIM = 1000`
How do the posteriors change? 